# 1 — Build the modelling datasets

Raw MATHia transaction exports to `data.pkl` + `problem_info.csv`, per workspace.

Five stages: keep the transactions worth modelling; derive problem subtypes and
per-problem conditional KCs from the problem metadata; build the Q-matrix; assemble
per-student response trajectories; write the per-(student, problem) side table.

*Stages 3 and 4 each define a function called `build`, and stages 1 and 4 each define
one called `summarize`. Since everything here shares one namespace, stage 3's is
`build_qmatrix` and stage 4's are `build_trajectories` and `summarize_trajectories`.*

## Imports

In [ ]:
from dataclasses import dataclass
from dataclasses import dataclass, field
from pathlib import Path
from typing import Dict, List, Sequence, Tuple
from typing import Dict, List, Tuple
import csv
import gc
import os
import pickle

import numpy as np
import pandas as pd

## Paths

In [ ]:
# Paths. Everything lives inside this folder, so the notebook is self-contained.
NOTEBOOK_DIR = Path.cwd()

DATASET_DIR = Path(os.environ.get("PC_DATASET_DIR", NOTEBOOK_DIR / "dataset"))
OUTPUT_DIR = Path(os.environ.get("PC_OUTPUT_DIR", NOTEBOOK_DIR / "outputs"))
RESULTS_DIR = Path(os.environ.get("PC_RESULTS_DIR", NOTEBOOK_DIR / "results"))


def workspace_dir(workspace, create=False):
    """Output directory holding one workspace's data.pkl and problem_info.csv."""
    path = OUTPUT_DIR / workspace
    if create:
        path.mkdir(parents=True, exist_ok=True)
    return path


print(f"dataset: {DATASET_DIR}\noutputs: {OUTPUT_DIR}\nresults: {RESULTS_DIR}")

## Workspace configuration

Shared constants and the per-workspace configuration.

`ratio_proportion_change3` and `ratio_proportion_change4` run the same pipeline; this
module holds everything that differs between them, so the stage code stays generic.

In [ ]:
# ── Structural steps ──────────────────────────────────────────────────────────────────
# Steps MATHia renders without a KC of its own. Both workspaces share these 11; change4
# adds three more.
STRUCTURAL_STEPS = [
    "OptionalTask_1", "EquationAnswer", "NumeratorFactor", "DenominatorFactor",
    "OptionalTask_2", "FirstRow1:1", "FirstRow1:2", "FirstRow2:1", "FirstRow2:2",
    "SecondRow", "ThirdRow",
]
CHANGE4_EXTRA_STRUCTURAL_STEPS = [
    "PercentChange", "NumeratorLabel1", "DenominatorLabel1",
]

# The two optional-task paths. OPT_STEP*[0] is the entry step, the rest are its substeps.
OPT_STEP1 = ["OptionalTask_1", "EquationAnswer", "NumeratorFactor", "DenominatorFactor"]
OPT_STEP2 = ["OptionalTask_2", "FirstRow1:1", "FirstRow1:2", "FirstRow2:1", "FirstRow2:2",
             "SecondRow", "ThirdRow"]
OPT_ALL_STEPS = set(OPT_STEP1 + OPT_STEP2)
OPT_SUBSTEPS = set(OPT_STEP1[1:] + OPT_STEP2[1:])

ER_PATH_STEPS = frozenset(OPT_STEP1)              # equivalent-ratio (fraction factor)
ME_PATH_STEPS = frozenset(OPT_STEP2)              # means-and-extremes

# ── KCs borrowed for KC-less structural steps ─────────────────────────────────────────
# Labels are taken verbatim from the prop1/prop2 workspaces so a skill keeps one name
# across all four workspaces.
STRUCTURAL_KC_CHANGE3 = {
    "FirstRow1:1": "enter first extreme in equation-1",
    "FirstRow1:2": "enter second extreme in equation-1",
    "FirstRow2:1": "enter first mean in equation-1",
    "FirstRow2:2": "enter second mean in equation-1",
    "SecondRow":   "calculate product of means or extremes-1",
}
STRUCTURAL_KC_CHANGE4 = {
    **STRUCTURAL_KC_CHANGE3,
    "PercentChange":        "identify percent change as increase or decrease-1",
    "NumeratorLabel1":      "enter proportion label in numerator-1",
    "DenominatorLabel1":    "enter proportion label in denominator-1",
    "DenominatorQuantity1": "enter given original amount in proportion-1",
}

# ── Conditional structural KCs ────────────────────────────────────────────────────────
# These four steps get their KC per problem, from the problem's numbers (see the metadata stage).
NUMFACTOR_INT_KC   = "enter numerator of form of 1-1 for integer factor"
NUMFACTOR_FRAC_KC  = "enter numerator of form of 1-1 for fractional factor"
DENFACTOR_INT_KC   = "enter denominator of form of 1-1 for integer factor"
DENFACTOR_FRAC_KC  = "enter denominator of form of 1-1 for fractional factor"
EQANSWER_PART_KC   = "calculate part in proportion with fractions-1"
EQANSWER_TOTAL_KC  = "calculate total in proportion with fractions-1"
THIRDROW_SIMPLE_KC = "calculate solution with means and extremes using simple numbers-1"
THIRDROW_DIFF_KC   = "calculate solution with means and extremes using difficult numbers-1"

STRUCTURAL_KC_CONDITIONAL_STEPS = {
    "NumeratorFactor", "DenominatorFactor", "EquationAnswer", "ThirdRow",
}

# ── Synthetic KCs appended to the Q-matrix ────────────────────────────────────────────
# Recognize-*: did the student take the path the problem's numbers call for?
RECOGNIZE_ER_KC = RECOGNIZE_ER_STEP = "Recognize-ER"
RECOGNIZE_ME_KC = RECOGNIZE_ME_STEP = "Recognize-ME"
# SelectOptimalStrategy: the same judgement as one observation per problem, appended at the
# end; the BeforeFA variant places it just before the final answer instead.
SOS_KC = SOS_STEP = "SelectOptimalStrategy"
SOS_BFA_KC = SOS_BFA_STEP = "SelectOptimalStrategyBeforeFA"

# ── Outcomes ──────────────────────────────────────────────────────────────────────────
GENUINE_OUTCOMES = {"OK", "ERROR"}
CORRECT_OUTCOME = "OK"

# ── Metadata scenario fields ──────────────────────────────────────────────────────────
# A subtype is the null pattern over these fields: which quantities the problem withholds.
CHANGE_FIELDS = [
    ("ppc-scenario_initial-amount", "initial"),
    ("ppc-scenario_final-amount",   "final"),
    ("ppc-scenario_change-amount",  "change"),
    ("ppc-scenario_percent-change", "pct"),
]
PROP_FIELDS = [
    ("ppc-scenario_percent",      "pct"),
    ("ppc-scenario_total-amount", "total"),
    ("ppc-scenario_part-amount",  "part"),
]

# change3 problem types, read off the metadata null pattern.
PROB_TYPE_CHANGE_AMOUNT     = "change_amount"      # pct given, change withheld
PROB_TYPE_CHANGE_PERCENT_IF = "change_percent_IF"  # initial + final given
PROB_TYPE_CHANGE_PERCENT_IC = "change_percent_IC"  # initial + change given
PROB_TYPE_CHANGE_PERCENT_FC = "change_percent_FC"  # final + change given

# Columns the transaction reconstruction writes back out, in MATHia's order.
MATHIA_COLUMNS = [
    "Anon Student Id", "Time", "Problem Name", "Step Name", "Attempt At Step",
    "Outcome", "Action", "Help Level", "KC Model(MATHia)",
    "CF (Skill Previous p-Known)", "CF (Skill New p-Known)", "CF (Etalon)",
    "CF (Is StepByStep)", "CF (Encounter)", "CF (Is Review Mode)",
    "CF (Is Autofilled)", "CF (Anon Class Id)", "CF (Anon School Id)",
    "CF (Workspace Progress Status)",
]


@dataclass(frozen=True)
class Workspace:
    """One MATHia workspace and the choices its pipeline run makes."""

    name: str
    dataset_file: str                       # raw transaction export, in DATASET_DIR
    metadata_file: str                      # problem text + scenario metadata
    structural_steps: List[str]             # KC-less steps, excluded from mastery skills
    structural_kc: Dict[str, str]           # step -> borrowed KC, unconditional steps
    metadata_columns: List[str]             # scenario columns to read
    prob_type_source: str                   # "metadata" (change3) or "steps" (change4)
    proportion_class_subtypes: bool = False  # split subtypes by problem-class (change4)
    coerce_metadata_numeric: bool = False   # some exports store scenario values as text
    # Steps exempt from the "mastery steps must carry a KC" rule, and the step whose
    # presence marks the problems where the exemption applies.
    mastery_kc_exempt: Tuple[Tuple[str, str], ...] = ()
    min_problems: int = 4                   # per-student problem count bounds
    max_problems: int = 40
    columns: List[str] = field(default_factory=lambda: list(MATHIA_COLUMNS))

    @property
    def structural_kc_steps(self) -> set:
        """Structural steps that end up in the Q-matrix, flat map plus conditional."""
        return set(self.structural_kc) | STRUCTURAL_KC_CONDITIONAL_STEPS


CHANGE3 = Workspace(
    name="ratio_proportion_change3",
    dataset_file="MATHia_2223_deidentified_ratio_proportion_change3_large_sample.csv",
    metadata_file="ratio_proportion_change3_all_problems_text_and_metadata.csv",
    structural_steps=STRUCTURAL_STEPS,
    structural_kc=STRUCTURAL_KC_CHANGE3,
    metadata_columns=["lms-id"] + [c for c, _ in CHANGE_FIELDS],
    prob_type_source="metadata",
)

CHANGE4 = Workspace(
    name="ratio_proportion_change4",
    dataset_file="MATHia_2223_deidentified_ratio_proportion_change4_large_sample.csv",
    metadata_file="ratio_proportion_change4_all_problems_text_and_metadata_corrected.csv",
    structural_steps=STRUCTURAL_STEPS + CHANGE4_EXTRA_STRUCTURAL_STEPS,
    structural_kc=STRUCTURAL_KC_CHANGE4,
    metadata_columns=(["lms-id", "problem-class"]
                      + [c for c, _ in CHANGE_FIELDS] + [c for c, _ in PROP_FIELDS]),
    prob_type_source="steps",
    proportion_class_subtypes=True,
    coerce_metadata_numeric=True,
    # DenominatorQuantity1 is KC-less in change4's _percentChange problems.
    mastery_kc_exempt=(("DenominatorQuantity1", "PercentChange"),),
)

WORKSPACES: Dict[str, Workspace] = {ws.name: ws for ws in (CHANGE3, CHANGE4)}

# Accept the short names used on the command line and in the notebooks.
ALIASES = {"change3": CHANGE3.name, "change4": CHANGE4.name}


def get_workspace(name: str) -> Workspace:
    """Look up a workspace by full name or short alias."""
    key = ALIASES.get(name, name)
    if key not in WORKSPACES:
        raise KeyError(f"unknown workspace {name!r}; "
                       f"choose from {sorted(WORKSPACES) + sorted(ALIASES)}")
    return WORKSPACES[key]


def compute_label_opt(step_set: Sequence[str]) -> int:
    """Classify optional-task engagement: 0 = none ... 8 = both paths fully completed."""
    step_set = set(step_set)
    all_opt1 = all(opt in step_set for opt in OPT_STEP1)
    any_opt1 = any(opt in step_set for opt in OPT_STEP1[1:])
    all_opt2 = all(opt in step_set for opt in OPT_STEP2)
    any_opt2 = any(opt in step_set for opt in OPT_STEP2[1:])
    label = 0
    if any_opt1:
        label = 2
    if all_opt1:
        label = 1
    if any_opt2:
        label = 4
    if all_opt2:
        label = 3
    if any_opt1 and any_opt2:
        label = 5
    if any_opt1 and all_opt2:
        label = 6
    if all_opt1 and any_opt2:
        label = 7
    if all_opt1 and all_opt2:
        label = 8
    return label


def recognize_response(er_me: int, has_er: bool, has_me: bool) -> int:
    """1 if the student took only the path the problem calls for, 0 if not, -1 if neither."""
    if not (has_er or has_me):
        return -1
    if er_me == 0:
        return 1 if (has_er and not has_me) else 0
    if er_me == 1:
        return 1 if (has_me and not has_er) else 0
    return -1

## Stage 1 — preprocess: the transactions worth modelling

Stage 1 -- load the raw MATHia transaction export and apply the inclusion filters.

In [ ]:
CHUNK_SIZE = 1_000_000


def parse_is_integer(etalon) -> int:
    """0 if the etalon value is a whole number, 1 if fractional, -1 if unparseable.

    The NumeratorFactor etalon is what tells ER (integer factor) apart from ME problems.
    """
    if pd.isna(etalon):
        return -1
    text = str(etalon)
    if "value=" in text:
        text = text.split("value=")[1].rstrip("}").strip()
    try:
        value = float(text)
    except (ValueError, TypeError):
        return -1
    return 0 if value == int(value) else 1


def _add_er_me_column(df: pd.DataFrame, verbose: bool) -> pd.DataFrame:
    """Attach ER_ME per (student, problem), falling back to the problem-level majority."""
    numerator = df[(df["Step Name"] == "NumeratorFactor") & (df["Action"] == "Attempt")].copy()
    numerator["ER_ME"] = numerator["CF (Etalon)"].apply(parse_is_integer)
    per_sp = (numerator.groupby(["Anon Student Id", "Problem Name"])["ER_ME"]
              .first().reset_index())
    df = df.merge(per_sp, on=["Anon Student Id", "Problem Name"], how="left")

    fallback = (per_sp[per_sp["ER_ME"] >= 0]
                .groupby("Problem Name")["ER_ME"]
                .agg(lambda x: x.mode().iloc[0])
                .to_dict())
    df["ER_ME"] = df.apply(
        lambda r: fallback.get(r["Problem Name"], -1)
        if pd.isna(r["ER_ME"]) or r["ER_ME"] == -1 else int(r["ER_ME"]),
        axis=1,
    ).astype(int)

    if verbose:
        print(f"ER_ME distribution: "
              f"{df.groupby('ER_ME')['Anon Student Id'].nunique().to_dict()}")
    del numerator, per_sp
    gc.collect()
    return df


def _sp_pairs(df: pd.DataFrame) -> set:
    return set(map(tuple, df[["Anon Student Id", "Problem Name"]].drop_duplicates().values))


def _valid_student_problems(df: pd.DataFrame, ws: Workspace, verbose: bool) -> set:
    """The (student, problem) pairs that pass every per-attempt inclusion rule."""
    expected_steps = ws.structural_steps
    all_sp = _sp_pairs(df)
    sp_group = df.groupby(["Anon Student Id", "Problem Name"])

    # At least 5 distinct steps attempted.
    step_counts = sp_group["Step Name"].nunique()
    keep_steps = set(step_counts[step_counts >= 5].index)

    # The problem was actually finished: a correct FinalAnswer and a correct Done.
    final_ok = df[(df["Step Name"] == "FinalAnswer") & (df["Outcome"] == "OK")]
    keep_final = _sp_pairs(final_ok)
    done_ok = df[(df["Action"] == "Done") & (df["Outcome"] == "OK")]
    keep_done = _sp_pairs(done_ok)

    # Structural steps must stay KC-less; a KC there means the export disagrees with the
    # workspace's step model.
    structural = df[df["Step Name"].isin(expected_steps)]
    keep_kc = all_sp - _sp_pairs(structural[structural["KC Model(MATHia)"].notna()])

    # Mirror image: the first attempt on every mastery step must carry a KC.
    attempts = df[df["Action"] == "Attempt"]
    mastery = attempts[~attempts["Step Name"].isin(expected_steps)]
    if mastery.empty:
        bad_mastery = set()
    else:
        mastery_first = (mastery
                         .sort_values(["Anon Student Id", "Problem Name", "Step Name", "Time"])
                         .groupby(["Anon Student Id", "Problem Name", "Step Name"])
                         .first().reset_index())
        null_kc = mastery_first["KC Model(MATHia)"].isna()
        for step, marker_step in ws.mastery_kc_exempt:
            marked_problems = set(attempts.loc[attempts["Step Name"] == marker_step,
                                               "Problem Name"].unique())
            null_kc &= ~((mastery_first["Step Name"] == step)
                         & mastery_first["Problem Name"].isin(marked_problems))
        bad_mastery = set(map(tuple, mastery_first.loc[
            null_kc, ["Anon Student Id", "Problem Name"]].drop_duplicates().values))
    if verbose:
        print(f"Mastery-KC rule: dropped {len(bad_mastery)} of {len(all_sp)} "
              f"(student, problem) pairs with null-KC mastery-step first attempts")

    return keep_steps & keep_final & keep_done & keep_kc & (all_sp - bad_mastery)


def load_and_filter(dataset_path, ws: Workspace, verbose: bool = True) -> pd.DataFrame:
    """Read the raw export and return the transactions the rest of the pipeline models."""
    chunks = []
    for chunk in pd.read_csv(dataset_path, sep=",", header=0,
                             iterator=True, chunksize=CHUNK_SIZE):
        # Tutor-driven rows are not evidence about the student: step-by-step scaffolding,
        # re-encounters of the same problem, and review mode.
        chunks.append(chunk[(chunk["CF (Is StepByStep)"] == False)  # noqa: E712
                            & (chunk["CF (Encounter)"] == 0)
                            & (chunk["CF (Is Review Mode)"] == -1)])
    df = pd.concat(chunks, ignore_index=True)
    del chunks
    gc.collect()
    if verbose:
        print(f"After initial filters: {df.shape}")

    df = _add_er_me_column(df, verbose)

    df = df[df["CF (Is Autofilled)"] == False]  # noqa: E712
    if verbose:
        print(f"After autofilled filter: {df.shape}")

    # A student in two classes cannot be attributed to one teacher's condition.
    if "CF (Anon Class Id)" in df.columns:
        classes = df.groupby("Anon Student Id")["CF (Anon Class Id)"].nunique()
        df = df[df["Anon Student Id"].isin(classes[classes == 1].index)]
        if verbose:
            print(f"After drop multi-class students: {df.shape}")

    valid_sp = _valid_student_problems(df, ws, verbose)
    pairs = list(zip(df["Anon Student Id"], df["Problem Name"]))
    df = df[pd.Series([sp in valid_sp for sp in pairs], index=df.index)]
    if verbose:
        print(f"After student-problem filters: {df.shape}")

    per_student = df.groupby("Anon Student Id")["Problem Name"].nunique()
    keep = per_student[(per_student >= ws.min_problems)
                       & (per_student <= ws.max_problems)].index
    df = df[df["Anon Student Id"].isin(keep)]
    if verbose:
        print(f"After per-student #problem filter ({ws.min_problems}-{ws.max_problems}): "
              f"{df.shape}")
        summarize(df)
    gc.collect()
    return df


def summarize(df: pd.DataFrame) -> None:
    """Print the counts worth eyeballing before the Q-matrix is built."""
    kcs = sorted(df["KC Model(MATHia)"].dropna().unique())
    print(f"\n#Students: {df['Anon Student Id'].nunique()}")
    print(f"#Problems: {df['Problem Name'].nunique()}")
    print(f"#Steps:    {df['Step Name'].nunique()}")
    print(f"#KCs:      {len(kcs)}")
    print(f"KCs: {kcs}")


def problem_er_me(df: pd.DataFrame) -> dict:
    """Problem -> ER (0) / ME (1), by majority over the students who attempted it."""
    return (df[df["ER_ME"] >= 0]
            .groupby("Problem Name")["ER_ME"]
            .agg(lambda x: int(x.mode().iloc[0]))
            .to_dict())


def school_map(df: pd.DataFrame) -> dict:
    """Student -> school id."""
    if "CF (Anon School Id)" not in df.columns:
        return {}
    return (df[["Anon Student Id", "CF (Anon School Id)"]]
            .drop_duplicates("Anon Student Id")
            .set_index("Anon Student Id")["CF (Anon School Id)"]
            .to_dict())


def included_steps_mask(frame: pd.DataFrame, ws: Workspace) -> pd.Series:
    """True for steps that carry a modelled skill: mastery steps plus mapped structural ones."""
    not_structural = ~frame["Step Name"].isin(ws.structural_steps)
    mapped_structural = frame["Step Name"].isin(ws.structural_kc_steps)
    return not_structural | mapped_structural


def problem_step_sets(df: pd.DataFrame) -> dict:
    """Problem -> the set of step names any student attempted on it."""
    return (df[(df["Action"] == "Attempt") & df["Step Name"].notna()]
            .groupby("Problem Name")["Step Name"]
            .apply(lambda x: frozenset(x.unique()))
            .to_dict())


def path_info(df: pd.DataFrame, er_steps, me_steps) -> dict:
    """(student, problem) -> (touched an ER step, touched an ME step)."""
    attempts = df[df["Action"] == "Attempt"]
    on_path = attempts[attempts["Step Name"].isin(set(er_steps) | set(me_steps))]
    return {
        (student, problem): (bool(set(group["Step Name"]) & set(er_steps)),
                             bool(set(group["Step Name"]) & set(me_steps)))
        for (student, problem), group in on_path.groupby(["Anon Student Id", "Problem Name"])
    }


def problem_order(df: pd.DataFrame, ws: Workspace) -> pd.DataFrame:
    """Chronological problem index t per student, over modelled steps only."""
    attempts = df[df["Action"] == "Attempt"].copy()
    attempts = attempts[included_steps_mask(attempts, ws)]
    order = (attempts.groupby(["Anon Student Id", "Problem Name"])["Time"]
             .min().reset_index()
             .sort_values(["Anon Student Id", "Time"]))
    order["t"] = order.groupby("Anon Student Id").cumcount()
    return order[["Anon Student Id", "Problem Name", "t"]]


def etalon_map(df: pd.DataFrame) -> dict:
    """Problem -> the NumeratorFactor etalon value, as exported."""
    numerator = df[(df["Step Name"] == "NumeratorFactor") & (df["Action"] == "Attempt")]
    values = {}
    for problem, group in numerator.groupby("Problem Name"):
        for etalon in group["CF (Etalon)"]:
            if pd.notna(etalon) and "value=" in str(etalon):
                values[problem] = str(etalon).split("value=")[1].rstrip("}").strip()
                break
    return values

## Stage 2 — problem metadata: subtypes and conditional KCs

Stage 2 -- problem metadata: subtypes, per-problem conditional KCs, problem types.

Four structural steps take a different KC depending on the problem's numbers, so their KC
cannot be read off the step name alone. Which KC applies is decided here, from the
scenario values in the workspace's metadata CSV, and the step is renamed accordingly
(`NumeratorFactor` -> `NumeratorFactor-int` / `-frac`) so the Q-matrix can tell them apart.

In [ ]:
PROPORTION_CLASS = "_percentProportion"


def load_metadata(path, ws: Workspace, verbose: bool = True) -> pd.DataFrame:
    """Read the metadata CSV, keep the scenario columns, and derive the subtype column."""
    raw = pd.read_csv(path)
    # Checked rather than tolerated: a subtype is the *null pattern* over the scenario
    # fields, so a column that is absent reads exactly like a quantity the problem
    # withholds. Skipping a missing column would quietly change the subtypes, the
    # Q-matrix and (change3) the problem types instead of failing.
    missing = [c for c in ws.metadata_columns if c not in raw.columns]
    if missing:
        raise RuntimeError(
            f"{path} is missing {len(missing)} column(s) the {ws.name} build reads: "
            f"{', '.join(missing)}")
    meta = raw[list(ws.metadata_columns)].copy()

    if ws.coerce_metadata_numeric:
        for col in [c for c in meta.columns if c.startswith("ppc-scenario_")]:
            meta[col] = pd.to_numeric(meta[col], errors="coerce")

    meta["subtype"] = _subtype_column(meta, ws)
    if verbose:
        print("Subtype distribution:")
        print(meta["subtype"].value_counts().to_string())
        print(f"\nTotal problems in metadata: {len(meta)}")
    return meta


def _subtype_column(meta: pd.DataFrame, ws: Workspace) -> pd.Series:
    """A subtype is the null pattern over the scenario fields: what the problem withholds."""
    has_class = ws.proportion_class_subtypes and "problem-class" in meta.columns

    def subtype(row):
        if has_class and row["problem-class"] == PROPORTION_CLASS:
            bits = [label for col, label in PROP_FIELDS
                    if col in meta.columns and pd.isna(row[col])]
            return "prop|" + "+".join(bits) if bits else "prop|none"
        bits = [label for col, label in CHANGE_FIELDS
                if col in meta.columns and pd.isna(row[col])]
        return "chg|" + "+".join(bits) if bits else "chg|none"

    return meta.apply(subtype, axis=1)


def subtype_map(meta: pd.DataFrame) -> dict:
    """Problem -> subtype."""
    return meta.set_index("lms-id")["subtype"].to_dict()


def scenario_map(path) -> dict:
    """Problem -> scenario tag, for the problem_info export."""
    scenarios = pd.read_csv(path, usecols=["lms-id", "scenario-tag"])
    return scenarios.set_index("lms-id")["scenario-tag"].to_dict()


def _is_integer_factor(initial) -> bool:
    """True when the form-of-1 factor between `initial` and 100 is a whole number."""
    if initial is None or pd.isna(initial) or initial == 0:
        return False
    factor = max(100 / initial, initial / 100)
    return abs(factor - round(factor)) < 1e-9


def _is_simple_thirdrow(divisor, cross) -> bool:
    """True when the means-and-extremes division stays easy (prop2's difficulty rule)."""
    if divisor is None or pd.isna(divisor):
        return True
    return divisor <= 10 or divisor == 100 or cross < 100


def _proportion_terms(row):
    """(factor base, ThirdRow divisor, ThirdRow cross-product, solve-for) for a proportion."""
    total = row.get("ppc-scenario_total-amount")
    pct = row.get("ppc-scenario_percent")
    part = row.get("ppc-scenario_part-amount")

    if pd.isna(part):
        solve_for = "part"
        divisor = total if pd.notna(total) else 0
        cross = abs(total * pct) if (pd.notna(total) and pd.notna(pct)) else 0
    elif pd.isna(total):
        solve_for = "total"
        divisor = 100
        cross = abs(part * 100) if pd.notna(part) else 0
    else:
        solve_for = "pct"
        divisor = total if pd.notna(total) else 0
        cross = abs(part * 100) if pd.notna(part) else 0
    return total, divisor, cross, solve_for


def _change_terms(row):
    """(factor base, ThirdRow divisor, ThirdRow cross-product, solve-for) for a change problem."""
    initial = row.get("ppc-scenario_initial-amount")
    final = row.get("ppc-scenario_final-amount")
    change = row.get("ppc-scenario_change-amount")
    pct = row.get("ppc-scenario_percent-change")

    # The percent is given and the change withheld -> the student solves for the change.
    solve_for = "change" if (pd.notna(pct) and pd.isna(change)) else "pct"

    # When the initial amount is withheld it is still recoverable, and the factor split
    # needs it.
    if pd.isna(initial) and pd.notna(final) and pd.notna(change):
        initial = final - change

    if solve_for == "change":
        divisor = 100
        cross = abs(initial * pct) if (pd.notna(initial) and pd.notna(pct)) else 0
    else:
        divisor = initial if pd.notna(initial) else 0
        cross = abs(change * 100) if pd.notna(change) else 0
    return initial, divisor, cross, solve_for


def build_structural_kc_maps(meta: pd.DataFrame, ws: Workspace, verbose: bool = True):
    """Assign the four conditional structural steps a KC per problem.

    Returns
    -------
    structural_kc_problem_map : {original step: {problem: (renamed step, KC)}}
    step_rename_map           : {(problem, original step): renamed step}
    """
    numfactor, denfactor, eqanswer, thirdrow = {}, {}, {}, {}

    for _, row in meta.iterrows():
        problem = row["lms-id"]
        if pd.isna(problem):
            continue

        is_proportion = (ws.proportion_class_subtypes
                         and row.get("problem-class") == PROPORTION_CLASS)
        factor_base, divisor, cross, solve_for = (
            _proportion_terms(row) if is_proportion else _change_terms(row))

        # NumeratorFactor / DenominatorFactor: is the form of 1 an integer factor?
        is_int = _is_integer_factor(factor_base)
        numfactor[problem] = (("NumeratorFactor-int", NUMFACTOR_INT_KC) if is_int
                              else ("NumeratorFactor-frac", NUMFACTOR_FRAC_KC))
        denfactor[problem] = (("DenominatorFactor-int", DENFACTOR_INT_KC) if is_int
                              else ("DenominatorFactor-frac", DENFACTOR_FRAC_KC))

        # EquationAnswer: solving for a part of the whole, or for the whole itself.
        solves_for_part = solve_for in ("part", "change")
        eqanswer[problem] = (("EquationAnswer", EQANSWER_PART_KC) if solves_for_part
                             else ("EquationAnswer", EQANSWER_TOTAL_KC))

        # ThirdRow: simple or difficult arithmetic.
        is_simple = _is_simple_thirdrow(divisor, cross)
        thirdrow[problem] = (("ThirdRow-simple", THIRDROW_SIMPLE_KC) if is_simple
                             else ("ThirdRow-difficult", THIRDROW_DIFF_KC))

    structural_kc_problem_map = {
        "NumeratorFactor": numfactor,
        "DenominatorFactor": denfactor,
        "EquationAnswer": eqanswer,
        "ThirdRow": thirdrow,
    }
    step_rename_map = {
        (problem, orig_step): new_step
        for orig_step, problem_map in structural_kc_problem_map.items()
        for problem, (new_step, _) in problem_map.items()
    }

    if verbose:
        n_int = sum(1 for step, _ in numfactor.values() if step.endswith("-int"))
        n_simple = sum(1 for step, _ in thirdrow.values() if step.endswith("-simple"))
        n_part = sum(1 for _, kc in eqanswer.values() if kc == EQANSWER_PART_KC)
        print(f"\nPer-problem structural KC map built ({len(numfactor)} problems):")
        print(f"  NumeratorFactor/DenominatorFactor: {n_int} int + "
              f"{len(numfactor) - n_int} frac")
        print(f"  ThirdRow: {n_simple} simple + {len(thirdrow) - n_simple} difficult")
        print(f"  EquationAnswer: {n_part} part + {len(eqanswer) - n_part} total")

    return structural_kc_problem_map, step_rename_map


def _classify_change_problem(row) -> str:
    """change3 problem type, from which two scenario quantities are given."""
    pct_null = pd.isna(row.get("ppc-scenario_percent-change"))
    change_null = pd.isna(row.get("ppc-scenario_change-amount"))
    initial_null = pd.isna(row.get("ppc-scenario_initial-amount"))
    final_null = pd.isna(row.get("ppc-scenario_final-amount"))
    if not pct_null and change_null:
        return PROB_TYPE_CHANGE_AMOUNT
    if not initial_null and not final_null:
        return PROB_TYPE_CHANGE_PERCENT_IF
    if not initial_null and not change_null:
        return PROB_TYPE_CHANGE_PERCENT_IC
    return PROB_TYPE_CHANGE_PERCENT_FC


def _classify_by_steps(steps: frozenset) -> str:
    """change4 problem type, from the steps the interface actually asked for."""
    if "NumeratorLabel1" in steps or "DenominatorLabel1" in steps:
        return "proportion_labeled"
    if "PercentChange" in steps and "FinalAnswerDirection" in steps:
        return "change_percent"
    if "PercentChange" in steps:
        return "change_amount"
    return "proportion_bare"


def build_prob_type_map(meta: pd.DataFrame, step_sets: dict, ws: Workspace,
                        verbose: bool = True) -> dict:
    """Problem -> problem type, using whichever source this workspace declares."""
    if ws.prob_type_source == "metadata":
        prob_types = {row["lms-id"]: _classify_change_problem(row)
                      for _, row in meta.iterrows() if pd.notna(row["lms-id"])}
    elif ws.prob_type_source == "steps":
        prob_types = {problem: _classify_by_steps(steps)
                      for problem, steps in step_sets.items()}
    else:
        raise ValueError(f"unknown prob_type_source {ws.prob_type_source!r}")

    if verbose:
        print(f"\nProblem type classification: {len(prob_types)} problems")
        for prob_type in sorted(set(prob_types.values())):
            print(f"  {prob_type}: "
                  f"{sum(1 for v in prob_types.values() if v == prob_type)}")
    return prob_types

## Stage 3 — the Q-matrix

Stage 3 -- the Q-matrix: which KCs each (step, subtype) cell exercises.

A cell's KC set is read off the data: over every student's first attempt at that step in
that subtype, keep the KCs MATHia credited on at least `KC_MIN_FRAC` of them. Cells that
share a KC set collapse into one Q-template, which is what the trajectories index into.

In [ ]:
# A KC has to explain at least this share of a cell's first attempts to be kept.
KC_MIN_FRAC = 0.05

# Above this many KCs, enumerating all 2^K mastery states stops being worth the memory.
MAX_MATERIALISED_K = 20

QMATRIX_GROUP_KEYS = ["Anon Student Id", "Problem Name", "Step Name"]
KC_COL = "KC Model(MATHia)"


@dataclass
class QMatrix:
    """The Q-matrix and everything derived from it that later stages need."""

    qmatrix: Dict[Tuple[str, str], List[int]]   # (step, subtype) -> KC indices
    kc_list: List[str]
    step_list: List[str]
    subtype_list: List[str]
    q_templates: np.ndarray                     # (n_templates, K) bool
    template_keys: List[List[Tuple[str, str]]]  # template -> the cells that use it
    template_idx_by_key: Dict[Tuple[str, str], int]
    valid_states: np.ndarray                    # None when K is too large to materialise
    dag_edges: List[Tuple[int, int]]            # prerequisite structure; empty in all-skills

    @property
    def K(self) -> int:
        return len(self.kc_list)


def first_attempts(df: pd.DataFrame, ws: Workspace,
                   structural_kc_problem_map: dict, verbose: bool = True) -> pd.DataFrame:
    """One row per (student, problem, step) first attempt, with the KC it exercises."""
    attempts = df[df["Action"] == "Attempt"]

    # Mastery steps come with their KC already assigned by MATHia.
    mastery = attempts[~attempts["Step Name"].isin(STRUCTURAL_STEPS)]
    mastery_first = (mastery.sort_values("Time").groupby(QMATRIX_GROUP_KEYS)
                     .first().reset_index()[QMATRIX_GROUP_KEYS + [KC_COL]])
    if verbose:
        print(f"Mastery first attempts: {len(mastery_first)}")

    # Structural steps need a KC assigned; rows that already carry one belong to the
    # mastery half and are dropped here to avoid double counting.
    structural = attempts[attempts["Step Name"].isin(ws.structural_kc_steps)].copy()
    struct_first = (structural.sort_values("Time").groupby(QMATRIX_GROUP_KEYS)
                    .first().reset_index())
    struct_first = struct_first[struct_first[KC_COL].isna()]

    # The four conditional steps: rename the step and take the KC from the problem map.
    for orig_step, problem_map in structural_kc_problem_map.items():
        mask = struct_first["Step Name"] == orig_step
        if not mask.any():
            continue
        renamed = {p: step for p, (step, _) in problem_map.items()}
        kcs = {p: kc for p, (_, kc) in problem_map.items()}
        struct_first.loc[mask, "Step Name"] = struct_first.loc[mask, "Problem Name"].map(renamed)
        struct_first.loc[mask, KC_COL] = struct_first.loc[mask, "Problem Name"].map(kcs)

    # Everything left takes its KC straight from the workspace's flat map.
    unassigned = struct_first[KC_COL].isna()
    struct_first.loc[unassigned, KC_COL] = (
        struct_first.loc[unassigned, "Step Name"].map(ws.structural_kc))

    struct_first = struct_first[QMATRIX_GROUP_KEYS + [KC_COL]]
    combined = pd.concat([mastery_first, struct_first], ignore_index=True)
    if verbose:
        print(f"Structural first attempts: {len(struct_first)}")
        print(f"Combined first attempts: {len(combined)}")
    return combined


def build_qmatrix(df: pd.DataFrame, meta: pd.DataFrame, ws: Workspace,
          structural_kc_problem_map: dict, prob_er_me: dict,
          verbose: bool = True) -> QMatrix:
    """Build the Q-matrix, append the synthetic KCs, and derive the Q-templates."""
    attempts = first_attempts(df, ws, structural_kc_problem_map, verbose)
    attempts = attempts.merge(meta[["lms-id", "subtype"]], left_on="Problem Name",
                              right_on="lms-id", how="left")
    attempts = attempts[attempts["subtype"].notna()]
    if verbose:
        print(f"After subtype merge: {len(attempts)}")

    counts = (attempts.groupby(["Step Name", "subtype", KC_COL])
              .size().reset_index(name="n"))
    kc_list = sorted(counts[KC_COL].dropna().unique())
    kc_index = {kc: i for i, kc in enumerate(kc_list)}

    qmatrix, multi_skill, steps, subtypes = {}, [], set(), set()
    for (step, subtype), block in counts.groupby(["Step Name", "subtype"]):
        block = block.dropna(subset=[KC_COL])
        if block.empty:
            continue
        block = block.assign(frac=block["n"] / float(block["n"].sum()))
        kept = block[block["frac"] >= KC_MIN_FRAC]
        if kept.empty:                       # keep the modal KC rather than an empty cell
            kept = block.sort_values("n", ascending=False).head(1)
        indices = sorted(kc_index[kc] for kc in kept[KC_COL])
        qmatrix[(step, subtype)] = indices
        steps.add(step)
        subtypes.add(subtype)
        if len(indices) > 1:
            multi_skill.append((step, subtype, [kc_list[i] for i in indices]))

    step_list = sorted(steps)
    subtype_list = sorted(subtypes)
    if verbose:
        print("\nQ-matrix built (before synthetic KCs):")
        print(f"  KCs      ({len(kc_list)}): {kc_list}")
        print(f"  Subtypes ({len(subtype_list)}): {subtype_list}")
        print(f"  Steps    ({len(step_list)}): {step_list}")
        print(f"  Multi-skill cells: {len(multi_skill)}")

    kc_list, step_list = _append_recognize_kcs(
        qmatrix, kc_list, step_list, subtype_list, meta, prob_er_me, ws, verbose)
    kc_list, step_list = _append_strategy_kcs(
        qmatrix, kc_list, step_list, subtype_list, verbose)

    q_templates, template_keys, template_idx_by_key = _build_templates(qmatrix, len(kc_list))
    valid_states = _enumerate_states(len(kc_list), verbose)

    qm = QMatrix(qmatrix=qmatrix, kc_list=kc_list, step_list=step_list,
                 subtype_list=subtype_list, q_templates=q_templates,
                 template_keys=template_keys, template_idx_by_key=template_idx_by_key,
                 valid_states=valid_states, dag_edges=[])
    if verbose:
        print_table(qm)
        print_templates(qm)
    return qm


def _append_recognize_kcs(qmatrix, kc_list, step_list, subtype_list, meta, prob_er_me,
                          ws, verbose):
    """One KC per path: did the student recognise which strategy the problem calls for?"""
    er_subtypes, me_subtypes = set(), set()
    for _, row in meta.iterrows():
        problem, subtype = row["lms-id"], row["subtype"]
        if pd.isna(problem) or pd.isna(subtype) or subtype not in subtype_list:
            continue
        er_me = prob_er_me.get(problem, -1)
        if er_me == 0:
            er_subtypes.add(subtype)
        elif er_me == 1:
            me_subtypes.add(subtype)

    er_idx, me_idx = len(kc_list), len(kc_list) + 1
    kc_list = kc_list + [RECOGNIZE_ER_KC, RECOGNIZE_ME_KC]
    for subtype in subtype_list:
        if subtype in er_subtypes:
            qmatrix[(RECOGNIZE_ER_STEP, subtype)] = [er_idx]
        if subtype in me_subtypes:
            qmatrix[(RECOGNIZE_ME_STEP, subtype)] = [me_idx]
    step_list = step_list + [RECOGNIZE_ER_STEP, RECOGNIZE_ME_STEP]

    if verbose:
        print(f"\nRecognize skills appended (KC{er_idx}: {RECOGNIZE_ER_KC}, "
              f"KC{me_idx}: {RECOGNIZE_ME_KC}):")
        print(f"  ER subtypes ({len(er_subtypes)}): {sorted(er_subtypes)}")
        print(f"  ME subtypes ({len(me_subtypes)}): {sorted(me_subtypes)}")
    return kc_list, step_list


def _append_strategy_kcs(qmatrix, kc_list, step_list, subtype_list, verbose):
    """Two views of the same strategy-choice event: one at the end, one before FinalAnswer."""
    for kc, step in ((SOS_KC, SOS_STEP), (SOS_BFA_KC, SOS_BFA_STEP)):
        index = len(kc_list)
        kc_list = kc_list + [kc]
        for subtype in subtype_list:
            qmatrix[(step, subtype)] = [index]
        step_list = step_list + [step]
        if verbose:
            print(f"\n{kc} appended (KC{index})")
    return kc_list, step_list


def _build_templates(qmatrix, K):
    """Collapse cells with identical KC sets into templates."""
    template_of_kcs, template_keys, template_idx_by_key = {}, [], {}
    for key in sorted(qmatrix.keys()):
        kcs = tuple(qmatrix[key])
        if kcs not in template_of_kcs:
            template_of_kcs[kcs] = len(template_of_kcs)
            template_keys.append([key])
        else:
            template_keys[template_of_kcs[kcs]].append(key)
        template_idx_by_key[key] = template_of_kcs[kcs]

    q_templates = np.zeros((len(template_of_kcs), K), dtype=bool)
    for kcs, template_id in template_of_kcs.items():
        for kc_idx in kcs:
            q_templates[template_id, kc_idx] = True
    return q_templates, template_keys, template_idx_by_key


def _enumerate_states(K, verbose):
    """All 2^K mastery states, or None when that is too large to hold."""
    if K > MAX_MATERIALISED_K:
        if verbose:
            print(f"\nValid states: S = 2^{K} = {2 ** K:,} (not materialised)")
        return None
    n_states = 1 << K
    states = np.zeros((n_states, K), dtype=bool)
    for col in range(K):
        period = 1 << (K - 1 - col)
        states[:, col] = np.tile(
            np.concatenate([np.zeros(period, dtype=bool), np.ones(period, dtype=bool)]),
            n_states // (2 * period))
    if verbose:
        print(f"\nValid states: S = {len(states)} (all 2^{K})")
    return states


def print_table(qm: QMatrix) -> None:
    """The Q-matrix as a step x subtype grid of KC indices, plus the KC legend."""
    print("\nQ-matrix (step x subtype -> KC indices):")
    print(f"  {'Step':<40}" + "".join(f"{s:<24}" for s in qm.subtype_list))
    for step in qm.step_list:
        row = f"  {step:<40}"
        for subtype in qm.subtype_list:
            indices = qm.qmatrix.get((step, subtype))
            cell = "+".join(f"{i}:KC{i}" for i in indices) if indices else "---"
            row += f"{cell:<24}"
        print(row)
    print("\nKC legend:")
    for i, kc in enumerate(qm.kc_list):
        print(f"  KC{i}: {kc}")
    print(f"\nFinal: K={qm.K} KCs, {len(qm.step_list)} steps, "
          f"{len(qm.subtype_list)} subtypes")


def print_templates(qm: QMatrix) -> None:
    """Template shape and any template that exercises more than one KC."""
    sizes = qm.q_templates.sum(axis=1)
    print(f"\nQ-templates: {qm.q_templates.shape}")
    print(f"  |R_q| min={int(sizes.min())}, max={int(sizes.max())}, "
          f"mean={float(sizes.mean()):.2f}")
    multi = np.where(sizes > 1)[0]
    if len(multi):
        print(f"  Multi-skill templates: {len(multi)}")
        for template_id in multi:
            kcs = [qm.kc_list[k] for k, on in enumerate(qm.q_templates[template_id]) if on]
            print(f"    template {template_id}: R_q={kcs}  "
                  f"used by {qm.template_keys[template_id]}")

## Stage 4 — trajectories

Stage 4 -- per-student trajectories, padded into the arrays that make up data.pkl.

In [ ]:
HELP_ACTIONS = {"Attempt", "Hint Request", "Hint Level Change"}
TRAJECTORY_GROUP_KEYS = ["Anon Student Id", "Problem Name", "Step Name"]
OUT_COLS = TRAJECTORY_GROUP_KEYS + ["Time", "response"]

MIN_PROBLEMS_PER_TRAJECTORY = 2


def first_genuine_attempts(df: pd.DataFrame, ws: Workspace, step_rename_map: dict,
                           verbose: bool = True) -> pd.DataFrame:
    """Score each step's first genuine encounter, counting hint-seeking as an error.

    A student who asks for a hint before attempting has not demonstrated the skill, so the
    opportunity is scored 0 rather than dropped. Four cases, by what the student did first:
    attempt it (A), open a substantive hint (B), open the first hint level and then
    escalate (C), or open the first hint level and then attempt (D).
    """
    relevant = df[df["Action"].isin(HELP_ACTIONS)].copy()
    relevant = relevant[included_steps_mask(relevant, ws)].sort_values("Time")
    first_action = relevant.groupby(TRAJECTORY_GROUP_KEYS).first().reset_index()

    is_hint = first_action["Action"] == "Hint Request"

    # A: attempted outright -- the outcome is the response.
    case_a = first_action[first_action["Action"] == "Attempt"].copy()
    case_a["response"] = (case_a["Outcome"] == CORRECT_OUTCOME).astype(np.int8)

    # B: went straight past the first hint level.
    case_b = first_action[is_hint & (first_action["Help Level"] >= 2)].copy()
    case_b["response"] = np.int8(0)

    # C/D: opened the first hint level -- the next action decides.
    hl1_keys = set(first_action.loc[is_hint & (first_action["Help Level"] < 2), TRAJECTORY_GROUP_KEYS]
                   .apply(tuple, axis=1))
    empty = pd.DataFrame(columns=OUT_COLS)
    escalated = direct_attempt = no_followup = empty

    if hl1_keys:
        hl1_rows = relevant[relevant.set_index(TRAJECTORY_GROUP_KEYS).index.isin(hl1_keys)]
        next_action = (hl1_rows[hl1_rows["Action"] != "Hint Request"]
                       .sort_values("Time").groupby(TRAJECTORY_GROUP_KEYS).first().reset_index())

        escalated = next_action[next_action["Action"] == "Hint Level Change"].copy()
        escalated["response"] = np.int8(0)

        direct_attempt = next_action[next_action["Action"] == "Attempt"].copy()
        direct_attempt["response"] = (
            direct_attempt["Outcome"] == CORRECT_OUTCOME).astype(np.int8)

        # Never came back to the step: no evidence of the skill.
        unresolved = hl1_keys - set(next_action[TRAJECTORY_GROUP_KEYS].apply(tuple, axis=1))
        if unresolved:
            no_followup = first_action[
                first_action[TRAJECTORY_GROUP_KEYS].apply(tuple, axis=1).isin(unresolved)].copy()
            no_followup["response"] = np.int8(0)

    parts = [case_a, case_b, escalated, direct_attempt, no_followup]
    genuine = pd.concat([p[OUT_COLS] for p in parts if len(p)], ignore_index=True)

    if verbose:
        scored_zero = len(case_b) + len(escalated) + len(no_followup)
        print(f"Hint-aware first genuine attempts: {len(genuine)}")
        print(f"  Case A (direct attempt):  {len(case_a)}")
        print(f"  Case B (HL>=2 first):     {len(case_b)}")
        print(f"  Case C (HL1->escalated):  {len(escalated)}")
        print(f"  Case D (HL1->attempt):    {len(direct_attempt)}")
        print(f"  No follow-up:             {len(no_followup)}")
        print(f"  Steps scored response=0 from hints: {scored_zero}")

    # The conditional structural steps carry their per-problem name from here on.
    if step_rename_map:
        genuine["Step Name"] = [
            step_rename_map.get((problem, step), step)
            for problem, step in zip(genuine["Problem Name"], genuine["Step Name"])
        ]
        if verbose:
            print(f"Applied step_rename_map: {len(step_rename_map)} (problem, step) entries")

    del relevant, first_action
    gc.collect()
    return genuine


def strategy_labels(df: pd.DataFrame, prob_er_me: dict, verbose: bool = True) -> dict:
    """(student, problem) -> did the student engage the path the problem calls for?

    1 when the optional-task engagement matches the problem's ER/ME type, 0 when the wrong
    path or both paths were engaged, -1 when no optional task was touched at all (no
    strategy was chosen, so there is nothing to score).
    """
    attempts = df[df["Action"] == "Attempt"]
    step_sets = (attempts.groupby(["Anon Student Id", "Problem Name"])["Step Name"]
                 .apply(set))

    labels = {}
    for key, step_set in step_sets.items():
        label = compute_label_opt(step_set)
        er_me = prob_er_me.get(key[1], -1)
        if label == 0:
            labels[key] = -1
        elif er_me == 0 and label in (1, 2):
            labels[key] = 1
        elif er_me == 1 and label in (3, 4):
            labels[key] = 1
        else:
            labels[key] = 0

    if verbose:
        n_pos = sum(1 for v in labels.values() if v == 1)
        n_neg = sum(1 for v in labels.values() if v == 0)
        n_unobserved = sum(1 for v in labels.values() if v == -1)
        print(f"SoS labels: {n_pos} pos + {n_neg} neg = {n_pos + n_neg} observed, "
              f"{n_unobserved} unobserved")
    return labels


def build_trajectories(prob_order: pd.DataFrame, genuine: pd.DataFrame, qm, subtype_map: dict,
          prob_type_map: dict, sos_info: dict, prob_er_me: dict, path_info: dict,
          school_map: dict, verbose: bool = True) -> list:
    """Assemble one response sequence per (student, problem), in chronological order."""
    by_student_problem = genuine.sort_values("Time").groupby(
        ["Anon Student Id", "Problem Name"], sort=False)

    trajectories = []
    for student_id, student_problems in prob_order.groupby("Anon Student Id"):
        problems = []
        for _, prow in student_problems.sort_values("t").iterrows():
            problem = prow["Problem Name"]
            subtype = subtype_map.get(problem)
            if subtype is None or subtype not in qm.subtype_list:
                continue
            try:
                steps = by_student_problem.get_group((student_id, problem))
            except KeyError:
                continue

            responses, templates, step_names = [], [], []
            for _, row in steps.iterrows():
                key = (row["Step Name"], subtype)
                if key not in qm.qmatrix:
                    continue
                responses.append(int(row["response"]))
                templates.append(qm.template_idx_by_key[key])
                step_names.append(row["Step Name"])
            if not responses:
                continue

            _insert_strategy_before_final_answer(
                responses, templates, step_names, subtype, qm,
                sos_info.get((student_id, problem), -1))
            _append_recognize(responses, templates, subtype, qm,
                              prob_er_me.get(problem, -1),
                              path_info.get((student_id, problem), (False, False)))
            _append_strategy(responses, templates, subtype, qm,
                             sos_info.get((student_id, problem), -1))

            problems.append({
                "problem_name": problem,
                "subtype": subtype,
                "prob_type": prob_type_map.get(problem, ""),
                "t": int(prow["t"]),
                "responses": np.array(responses, dtype=np.int8),
                "q_template_idx": np.array(templates, dtype=np.int16),
            })

        if len(problems) >= MIN_PROBLEMS_PER_TRAJECTORY:
            trajectories.append({"student_id": student_id,
                                 "school_id": school_map.get(student_id),
                                 "problems": problems})

    if verbose:
        print(f"Trajectories built: {len(trajectories)} students")
        print(f"Total problems: {sum(len(s['problems']) for s in trajectories)}")
    return trajectories


def _insert_strategy_before_final_answer(responses, templates, step_names, subtype, qm,
                                         sos_value):
    """Place the strategy-choice observation immediately before the last FinalAnswer."""
    if sos_value == -1:
        return
    key = (SOS_BFA_STEP, subtype)
    if key not in qm.template_idx_by_key:
        return
    final_answer_pos = None
    for i, step in enumerate(step_names):
        if step == "FinalAnswer":
            final_answer_pos = i
    if final_answer_pos is None:
        return
    responses.insert(final_answer_pos, int(sos_value))
    templates.insert(final_answer_pos, qm.template_idx_by_key[key])


def _append_recognize(responses, templates, subtype, qm, er_me, paths):
    """Append the recognition observation, if the student engaged either path."""
    has_er, has_me = paths
    if not (has_er or has_me):
        return
    if er_me == 0:
        key, response = (RECOGNIZE_ER_STEP, subtype), int(has_er and not has_me)
    elif er_me == 1:
        key, response = (RECOGNIZE_ME_STEP, subtype), int(has_me and not has_er)
    else:
        return
    if key in qm.template_idx_by_key:
        responses.append(response)
        templates.append(qm.template_idx_by_key[key])


def _append_strategy(responses, templates, subtype, qm, sos_value):
    """Append the end-of-problem strategy-choice observation, if it was observed."""
    key = (SOS_STEP, subtype)
    if sos_value != -1 and key in qm.template_idx_by_key:
        responses.append(int(sos_value))
        templates.append(qm.template_idx_by_key[key])


def pad_to_arrays(trajectories: list, qm, prob_type_list: list,
                  verbose: bool = True) -> dict:
    """Pad the ragged trajectories into the (N, T, Q) arrays data.pkl stores."""
    N = len(trajectories)
    T_max = max(len(s["problems"]) for s in trajectories)
    Q_max = max(len(p["responses"]) for s in trajectories for p in s["problems"])

    subtype_index = {s: i for i, s in enumerate(qm.subtype_list)}
    prob_type_index = {p: i for i, p in enumerate(prob_type_list)}

    responses = np.full((N, T_max, Q_max), -1, dtype=np.int8)
    q_template_idx = np.full((N, T_max, Q_max), -1, dtype=np.int16)
    response_mask = np.zeros((N, T_max, Q_max), dtype=bool)
    problem_mask = np.zeros((N, T_max), dtype=bool)
    subtypes = np.full((N, T_max), -1, dtype=np.int8)
    prob_types = np.full((N, T_max), -1, dtype=np.int8)
    n_problems = np.zeros(N, dtype=np.int32)

    for n, student in enumerate(trajectories):
        n_problems[n] = len(student["problems"])
        problem_mask[n, :n_problems[n]] = True
        for t, problem in enumerate(student["problems"]):
            width = len(problem["responses"])
            responses[n, t, :width] = problem["responses"]
            q_template_idx[n, t, :width] = problem["q_template_idx"]
            response_mask[n, t, :width] = True
            subtypes[n, t] = subtype_index.get(problem["subtype"], -1)
            prob_types[n, t] = prob_type_index.get(problem["prob_type"], -1)

    dataset = {
        "responses": responses,
        "q_template_idx": q_template_idx,
        "q_templates": qm.q_templates,
        "q_template_keys": qm.template_keys,
        "response_mask": response_mask,
        "problem_mask": problem_mask,
        "subtypes": subtypes,
        "prob_types": prob_types,
        "n_problems": n_problems,
        # Kept per student so downstream analyses can join on problem id across students.
        "problem_names": [[p["problem_name"] for p in s["problems"]] for s in trajectories],
        "student_ids": [s["student_id"] for s in trajectories],
        "school_ids": [s["school_id"] for s in trajectories],
        "subtype_list": qm.subtype_list,
        "prob_type_list": prob_type_list,
        "kc_list": qm.kc_list,
        "valid_states": qm.valid_states,
        "dag_edges": qm.dag_edges,
    }
    if verbose:
        summarize_trajectories(dataset, trajectories)
    return dataset


def summarize_trajectories(dataset: dict, trajectories: list) -> None:
    """Print the dataset's shape, accuracy and per-category problem counts."""
    observed = dataset["responses"][dataset["response_mask"]]
    n_problems = dataset["n_problems"]
    print("Dataset summary:")
    print(f"  N (students):     {len(dataset['student_ids'])}")
    print(f"  T_max (problems): {dataset['responses'].shape[1]}")
    print(f"  Q_max (steps):    {dataset['responses'].shape[2]}")
    print(f"  K (KCs):          {len(dataset['kc_list'])}")
    print(f"  responses shape:  {dataset['responses'].shape}")
    print(f"  q_templates:      {dataset['q_templates'].shape}")
    print(f"  Overall accuracy: {observed.mean() * 100:.1f}% "
          f"({observed.sum()}/{len(observed)} steps correct)")
    print(f"  Problems/student: min={n_problems.min()}, max={n_problems.max()}, "
          f"mean={n_problems.mean():.1f}")

    for field in ("subtype", "prob_type"):
        counts = {}
        for student in trajectories:
            for problem in student["problems"]:
                counts[problem[field]] = counts.get(problem[field], 0) + 1
        print(f"  Problems per {field}:")
        for key, count in sorted(counts.items()):
            print(f"    {key}: {count}")

## Stage 5 — the problem_info side table

Stage 5 -- problem_info.csv: one row per (student, problem), in solve order.

data.pkl carries what the models consume; this file carries everything else about the same
problem attempt -- timing, optional-task engagement, the full transaction token sequence.
The token sequence is lossless enough that notebook 2 can rebuild the raw
transactions from it.

In [ ]:
HEADER = [
    "school", "progress", "student", "count", "total_problems",
    "problem", "prob_type", "scenario", "er_me", "etalon_value",
    "label_opt", "all_opt_correct", "fa_correctness_after_opt", "n_tokenized_steps",
    "n_original_steps", "original_steps", "tokenized_steps", "kcs_skills",
    "new_skills", "fa_skill_index", "seq_time_vec", "opt_step_time",
    "non_opt_step_time", "actual_fa_opt_time", "tot_prob_time",
    "recognize_response",
]

# Columns the transaction token sequence is built from.
TRANSACTION_COLS = [
    "Anon Student Id", "Problem Name", "Step Name", "Action", "Attempt At Step",
    "Help Level", "Outcome", "Time", "KC Model(MATHia)",
    "CF (Skill Previous p-Known)", "CF (Skill New p-Known)",
]


def tokenize_steps(steps, actions, outcomes) -> str:
    """Collapse a transaction sequence into one token per step, up to the final answer.

    Non-optional steps become `<step>-<code>` where the code is the worst thing that
    happened on them: 0 = clean, 1 = hint used, 2 = wrong attempt. Optional-task steps keep
    their bare name. Returns "" for problems where no optional task was engaged, since the
    sequence exists to describe optional-task behaviour.
    """
    tokens = []
    final_answer_idx = 0
    opt_used = False
    final_answer_pending = True

    for step, action, outcome in zip(steps, actions, outcomes):
        previous_step = tokens[-1].split("-")[0] if tokens else ""
        new_step = not tokens or step != previous_step

        if new_step and step in OPT_ALL_STEPS:
            tokens.append(step)
            if step in OPT_SUBSTEPS:
                opt_used = True
            continue

        if action == "Attempt" and outcome != "OK":
            token = step + "-2"
        elif "Hint" in str(action):
            token = step + "-1"
        else:
            token = step + "-0"

        if new_step:
            if step == "FinalAnswer" and opt_used and final_answer_pending:
                final_answer_idx = len(tokens)
                final_answer_pending = False
            tokens.append(token)
        elif step not in OPT_ALL_STEPS and tokens[-1] < token:
            tokens[-1] = token          # keep the worst code seen on this step

    if not (opt_used and tokens):
        return ""
    head = tokens[:final_answer_idx + 1]
    head[-1] = "FinalAnswer"
    return "\t".join(head)


def time_info(steps, times):
    """Inter-step gaps, split by optional/non-optional, up to the post-optional answer.

    Returns (seq_time_vec, opt_step_time, non_opt_step_time, actual_fa_opt_time, total).
    `actual_fa_opt_time` is how long the student took from their last optional-task step to
    submitting the final answer.
    """
    seq, opt_times, non_opt_times = [], [], []
    opt_used = False
    final_answer_idx = last_opt_idx = -1

    for i, (step, time) in enumerate(zip(steps, times)):
        delta = (time - times[i - 1]) if i > 0 else 0
        if step in OPT_SUBSTEPS:
            opt_used = True
            last_opt_idx = i
            opt_times.append(delta)
        else:
            non_opt_times.append(delta)
        seq.append(delta)
        if opt_used and step == "FinalAnswer":
            final_answer_idx = i
            break

    fa_opt_time = 0
    if final_answer_idx != -1 and last_opt_idx != -1:
        fa_opt_time = times[final_answer_idx] - times[last_opt_idx]
    return seq, opt_times, non_opt_times, fa_opt_time, sum(seq)


def optional_task_flags(group: pd.DataFrame):
    """(every optional step right first time, final answer correct after optional work)."""
    opt_first = (group[group["Step Name"].isin(OPT_SUBSTEPS)]
                 .drop_duplicates("Step Name", keep="first"))
    all_correct = bool(not opt_first.empty and ((opt_first["Outcome"] == "OK")
                                                & (opt_first["Action"] == "Attempt")
                                                & (opt_first["Attempt At Step"] == 1)).all())
    correctness = 0
    if set(group["Step Name"]) & OPT_SUBSTEPS:
        final_answer = group[group["Step Name"] == "FinalAnswer"]
        if not final_answer.empty and final_answer.iloc[0]["Outcome"] == "OK":
            correctness = 1
    return all_correct, correctness


def export_problem_info(path, df: pd.DataFrame, prob_type_map: dict, scenario_map: dict,
           prob_er_me: dict, path_info: dict, etalon_map: dict,
           verbose: bool = True) -> int:
    """Write problem_info.csv and return the number of rows written."""
    genuine = df[(df["Action"] == "Attempt") & df["Outcome"].isin(GENUINE_OUTCOMES)]
    order = (genuine.groupby(["Anon Student Id", "Problem Name"])["Time"]
             .min().reset_index()
             .sort_values(["Anon Student Id", "Time"]))
    total_problems = order.groupby("Anon Student Id")["Problem Name"].nunique().to_dict()

    student_meta = (df.groupby("Anon Student Id")
                    .agg(school=("CF (Anon School Id)", "first"),
                         progress=("CF (Workspace Progress Status)", "first"))
                    .to_dict("index"))

    # p-Known snapshots are indexed by the workspace's own mastery KC list.
    kcs = sorted(df["KC Model(MATHia)"].dropna().unique())
    kc_index = {kc: i for i, kc in enumerate(kcs)}

    non_autofilled = df[df["CF (Is Autofilled)"] == False]  # noqa: E712
    columns = [c for c in TRANSACTION_COLS if c in non_autofilled.columns]
    groups = dict(tuple(non_autofilled[columns].groupby(["Anon Student Id", "Problem Name"])))

    n_rows = 0
    with open(path, "w", newline="") as handle:
        writer = csv.writer(handle)
        writer.writerow(HEADER)

        for student, student_problems in order.groupby("Anon Student Id"):
            meta = student_meta.get(student, {})
            for count, (_, prow) in enumerate(
                    student_problems.sort_values("Time").iterrows(), 1):
                problem = prow["Problem Name"]
                group = groups.get((student, problem))
                if group is None or group.empty:
                    continue
                group = group.sort_values("Time")
                row = _build_row(student, problem, count, group, meta,
                                 total_problems.get(student, 0), prob_type_map,
                                 scenario_map, prob_er_me, path_info, etalon_map,
                                 kcs, kc_index)
                writer.writerow(row)
                n_rows += 1

    if verbose:
        print(f"problem_info.csv saved -> {path}")
        print(f"  {n_rows} rows (student-problem pairs)")
    return n_rows


def _build_row(student, problem, count, group, meta, n_total, prob_type_map, scenario_map,
               prob_er_me, path_info, etalon_map, kcs, kc_index):
    er_me = prob_er_me.get(problem, -1)
    has_er, has_me = path_info.get((student, problem), (False, False))

    clean = group.dropna(subset=["Step Name"])
    step_names = list(clean["Step Name"])
    times = list(clean["Time"])
    step_set = set(step_names)

    # One record per raw transaction, in the order they happened.
    records = (clean["Step Name"].astype(str) + "-"
               + clean["Action"].astype(str) + "-"
               + clean["Attempt At Step"].astype(str) + "-"
               + clean["Help Level"].astype(str) + "-"
               + clean["Outcome"].astype(str) + "-"
               + clean["Time"].astype(str)).tolist()

    all_opt_correct, fa_correctness = optional_task_flags(clean)
    tokenized = tokenize_steps(step_names, list(clean["Action"]), list(clean["Outcome"]))
    seq_time, opt_time, non_opt_time, fa_opt_time, total_time = time_info(step_names, times)

    # Snapshot of MATHia's own mastery estimates before and after each attempt.
    prev_known = [0] * len(kcs)
    new_known = [0] * len(kcs)
    fa_skill_index = -1
    attempts = clean[(clean["Action"] == "Attempt") & clean["KC Model(MATHia)"].notna()]
    for kc, previous, updated, step in zip(attempts["KC Model(MATHia)"],
                                          attempts["CF (Skill Previous p-Known)"],
                                          attempts["CF (Skill New p-Known)"],
                                          attempts["Step Name"]):
        index = kc_index.get(kc)
        if index is not None:
            prev_known[index] = previous
            new_known[index] = updated
            if step == "FinalAnswer":
                fa_skill_index = index

    return [
        meta.get("school", ""), meta.get("progress", ""), student, count, n_total,
        problem, prob_type_map.get(problem, ""), scenario_map.get(problem, ""),
        "ME" if er_me == 1 else ("ER" if er_me == 0 else "UNK"),
        etalon_map.get(problem, ""),
        compute_label_opt(step_set), all_opt_correct, fa_correctness,
        len(step_set), len(records), "\t".join(records), tokenized,
        "\t".join(str(v) for v in prev_known),
        "\t".join(str(v) for v in new_known),
        fa_skill_index,
        "\t".join(str(v) for v in seq_time),
        "\t".join(str(v) for v in opt_time),
        "\t".join(str(v) for v in non_opt_time),
        fa_opt_time, total_time,
        recognize_response(er_me, has_er, has_me),
    ]

## Build and verify

In [ ]:
def build(ws, dataset_dir, out_dir, verbose=True):
    """Run every stage and write data.pkl + problem_info.csv."""
    dataset_path = dataset_dir / ws.dataset_file
    metadata_path = dataset_dir / ws.metadata_file
    for path in (dataset_path, metadata_path):
        if not path.exists():
            raise FileNotFoundError(f"{path} not found -- put the MATHia exports in "
                                    f"{dataset_dir}, keeping the file names above.")

    out_dir.mkdir(parents=True, exist_ok=True)
    if verbose:
        print(f"workspace: {ws.name}\ndataset:   {dataset_path}\n"
              f"metadata:  {metadata_path}\noutput:    {out_dir}\n")

    df = load_and_filter(dataset_path, ws, verbose)
    prob_er_me = problem_er_me(df)

    meta = load_metadata(metadata_path, ws, verbose)
    structural_kc_problem_map, step_rename_map = build_structural_kc_maps(meta, ws, verbose)
    prob_type_map = build_prob_type_map(meta, problem_step_sets(df), ws, verbose)

    qm = build_qmatrix(df, meta, ws, structural_kc_problem_map, prob_er_me, verbose)

    built = build_trajectories(
        prob_order=problem_order(df, ws),
        genuine=first_genuine_attempts(df, ws, step_rename_map, verbose),
        qm=qm, subtype_map=subtype_map(meta), prob_type_map=prob_type_map,
        sos_info=strategy_labels(df, prob_er_me, verbose), prob_er_me=prob_er_me,
        path_info=path_info(df, ER_PATH_STEPS, ME_PATH_STEPS),
        school_map=school_map(df), verbose=verbose)
    if not built:
        raise RuntimeError("no student survived the filters -- nothing to write")

    dataset = pad_to_arrays(built, qm, sorted(set(prob_type_map.values())), verbose)
    with open(out_dir / "data.pkl", "wb") as handle:
        pickle.dump(dataset, handle, protocol=pickle.HIGHEST_PROTOCOL)
    if verbose:
        print(f"\ndata.pkl saved -> {out_dir / 'data.pkl'}")

    export_problem_info(
        out_dir / "problem_info.csv", df, prob_type_map=prob_type_map,
        scenario_map=scenario_map(metadata_path), prob_er_me=prob_er_me,
        path_info=path_info(df, ER_PATH_STEPS, ME_PATH_STEPS),
        etalon_map=etalon_map(df), verbose=verbose)
    return out_dir


def verify(out_dir):
    """Reload both artefacts and print what notebook 3 will see."""
    with open(out_dir / "data.pkl", "rb") as handle:
        dataset = pickle.load(handle)

    print("\n=== data.pkl ===")
    for key in ("responses", "q_template_idx", "q_templates", "response_mask",
                "problem_mask", "subtypes", "prob_types", "n_problems"):
        print(f"  {key + ':':<16} {dataset[key].shape}")
    print(f"  {'students:':<16} {len(dataset['student_ids'])}")
    print(f"  {'KCs:':<16} {len(dataset['kc_list'])}")
    print(f"  prob_type_list:  {dataset['prob_type_list']}")
    observed = dataset["responses"][dataset["response_mask"]]
    print(f"  accuracy:        {observed.mean() * 100:.1f}% "
          f"({observed.sum()}/{len(observed)})")

    info = pd.read_csv(out_dir / "problem_info.csv")
    print(f"\n=== problem_info.csv ===\n  rows: {len(info):,}  "
          f"students: {info['student'].nunique():,}  "
          f"problems: {info['problem'].nunique():,}")

### Run

Point `DATASET_DIR` (or `PC_DATASET_DIR`) at the MATHia exports.

In [ ]:
for ws in (CHANGE3, CHANGE4):
    verify(build(ws, DATASET_DIR, workspace_dir(ws.name), verbose=True))